In [7]:
import numpy as np
from PIL import Image
import os
from collections import defaultdict
import random

class NaiveBayesImageClassifier:
    def __init__(self):
        self.class_stats = defaultdict(lambda: {
            'prior': 0,
            'mean_brightness': {'mean': 0, 'std': 0},
            'green_ratio': {'mean': 0, 'std': 0},
            'contrast': {'mean': 0, 'std': 0}
        })
        self.classes = []

    def extract_features(self, image_path):
        """Извлекает признаки изображения."""
        img = Image.open(image_path).convert('RGB')
        img_array = np.array(img) / 255.0 

        # Средняя яркость
        brightness = np.mean(img_array)

        # Доля зелёного (зелёный канал в RGB)
        green_ratio = np.mean(img_array[:, :, 1])

        # Контраст (стандартное отклонение яркости)
        contrast = np.std(img_array)

        return {
            'mean_brightness': brightness,
            'green_ratio': green_ratio,
            'contrast': contrast
        }

    def train_test_split(self, X, y, test_size=0.2, random_state=None):
        """Разделяет данные на обучающую и тестовую выборки."""
        if random_state is not None:
            random.seed(random_state)
            
        data = list(zip(X, y))
        random.shuffle(data)
        split_idx = int(len(data) * (1 - test_size))
        
        train_data = data[:split_idx]
        test_data = data[split_idx:]
        
        X_train = [x for x, y in train_data]
        y_train = [y for x, y in train_data]
        X_test = [x for x, y in test_data]
        y_test = [y for x, y in test_data]
        
        return X_train, X_test, y_train, y_test

    def fit(self, X_train, y_train):
        """Обучает модель на обучающих данных."""
        self.classes = list(set(y_train))
        features_by_class = defaultdict(list)

        # Сбор признаков для каждого класса
        for image_path, label in zip(X_train, y_train):
            features = self.extract_features(image_path)
            features_by_class[label].append(features)

        # Расчёт статистик
        for label in self.classes:
            features_list = features_by_class[label]
            self.class_stats[label]['prior'] = len(features_list) / len(X_train)

            for feature_name in ['mean_brightness', 'green_ratio', 'contrast']:
                values = [f[feature_name] for f in features_list]
                self.class_stats[label][feature_name]['mean'] = np.mean(values)
                self.class_stats[label][feature_name]['std'] = np.std(values) + 1e-6 

    def predict(self, image_path):
        """Предсказывает класс для одного изображения."""
        features = self.extract_features(image_path)
        best_class = None
        max_posterior = -np.inf

        for label in self.classes:
            stats = self.class_stats[label]
            log_posterior = np.log(stats['prior'])

            for feature_name, value in features.items():
                mean = stats[feature_name]['mean']
                std = stats[feature_name]['std']

                log_likelihood = -0.5 * np.log(2 * np.pi * std**2) - 0.5 * ((value - mean) / std)**2
                log_posterior += log_likelihood

            if log_posterior > max_posterior:
                max_posterior = log_posterior
                best_class = label

        return best_class

    def evaluate(self, X_test, y_test):
        """Оценивает точность модели на тестовых данных."""
        correct = 0
        for image_path, true_label in zip(X_test, y_test):
            pred_label = self.predict(image_path)
            if pred_label == true_label:
                correct += 1
        return correct / len(X_test)

def load_dataset(dataset_path):
    """Загружает все изображения из папок forest и desert."""
    X = []
    y = []
    
    for class_name in ['forest', 'desert']:
        class_dir = os.path.join(dataset_path, class_name)
        for img_file in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_file)
            X.append(img_path)
            y.append(class_name)
    
    return X, y

In [8]:
# Путь к папке с датасетом
DATASET_PATH = "origins/task_2"

# Загружаем все изображения
X, y = load_dataset(DATASET_PATH)

# Создаем и инициализируем классификатор
classifier = NaiveBayesImageClassifier()

# Разделяем данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = classifier.train_test_split(X, y, test_size=0.2, random_state=42)

# Обучаем модель
classifier.fit(X_train, y_train)

In [11]:
# Оцениваем точность на тестовых данных
accuracy = classifier.evaluate(X_test, y_test)
print(f"Точность модели: {accuracy * 100:.2f}%")

# Пример предсказания для нового изображения
test_image = "origins/task_2/test/test_desert_1.jpg"
predicted_class = classifier.predict(test_image)
print(f"Изображение {test_image} относится к классу: {predicted_class}")

Точность модели: 78.88%
Изображение origins/task_2/test/test_desert_1.jpg относится к классу: desert
